# Lab 3: Information Extraction


**Exercise 1.** Implement a trie that allows the operations *insert* and *find*. Add all the entities from the file *winners.txt* to the trie. Using this trie, perform entity recognition in the text file *example-nobel.txt*. The files are on Moodle.

You can use as starting point the following code: 

In [18]:
class TrieNode:
	"""A node in the trie structure"""

	def __init__(self, char):
		# the character stored in this node
		self.char = char

		# whether this can be the end of a word
		self.is_end = False

		# a dictionary of child nodes
		# keys are characters, values are nodes
		self.children = {}


class Trie(object):

	def __init__(self):
		"""Create an empty trie (only the root)."""
		# Root does not store any character.
		self.root = TrieNode("")

	def insert(self, word):
		"""Insert a word into the trie."""
		# We ignore None/empty strings to keep the trie consistent.
		if word is None:
			return
		word = str(word)
		if word == "":
			return

		node = self.root
		for ch in word:
			if ch not in node.children:
				node.children[ch] = TrieNode(ch)
			node = node.children[ch]
		# Mark that the path up to here corresponds to a complete word.
		node.is_end = True

	def find(self, x):
		"""Return (is_prefix, is_word).

		- is_prefix: True iff x matches a path in the trie (prefix exists)
		- is_word: True iff x is exactly a word inserted in the trie
		"""
		if x is None:
			return (False, False)
		x = str(x)
		# Empty string is always a prefix (the root), but not a word.
		if x == "":
			return (True, False)

		node = self.root
		for ch in x:
			if ch not in node.children:
				return (False, False)
			node = node.children[ch]
		return (True, node.is_end)


In [32]:
# 1. Chargement et construction du Trie
trie = Trie()
for name in open("data/winners.txt").read().splitlines():
    if name.strip():
        trie.insert(name.strip())

# 2. Reconnaissance (Longest Match) dans le texte sans tokenisation
text = open("data/example-nobel.txt").read()
mentions = []
i = 0
while i < len(text):
    node = trie.root
    best_match = None

    for j in range(i, len(text)):
        ch = text[j]
        if ch not in node.children:
            break
        node = node.children[ch]
        if node.is_end:
            best_match = (text[i : j + 1], i, j + 1)

    if best_match:
        mentions.append(best_match)
        i = best_match[2]  # On saute à la fin de l'entité
    else:
        i += 1

print(mentions)

[('Henry Kissinger', 68, 83), ('Le Duc Tho', 88, 98), ('Le Duc Tho', 653, 663), ('Yasser Arafat', 691, 704), ('Shimon Peres', 706, 718), ('Yitzhak Rabin', 724, 737), ('Barack Obama', 1108, 1120), ('Jimmy Carter', 1637, 1649), ('Al Gore', 1654, 1661)]


**Exercise 2.** Supposing that all entities are written such that they start with a capital letter, perform entity recognition in the text file *example-nobel.txt* using regular expression. You should also be able to capture with the regex that an entity can have 2 or more names, such as Le Duc Tho. For this, first follow this tutorial https://www.w3schools.com/python/python_regex.asp or https://docs.python.org/3/howto/regex.html on the **re library** in Python. When you have an idea for a regex, you can test it using this website https://regex101.com/ . Note the website even generates the corresponding Python code for you! You might need to read a bit about repeating a group https://www.regular-expressions.info/captureall.html .

In [30]:
# Solution exercise 2 (simple)

import re
from pathlib import Path

text = Path("data/example-nobel.txt").read_text(encoding="utf-8")

pattern = r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b"

mentions = [(m.group(0), m.start(), m.end()) for m in re.finditer(pattern, text)]

print("#mentions:", len(mentions))
print("#unique:", len(set(m[0] for m in mentions)))
print("mentions (entity, start, end):")
for ent, start, end in mentions:
	print(ent, start, end)

#mentions: 42
#unique: 32
mentions (entity, start, end):
Among 0 5
Nobel Peace Prizes 26 44
Henry Kissinger 68 83
Le Duc Tho 88 98
This 100 104
Norwegian Nobel Committee 135 160
Kissinger 170 179
North Vietnam 247 260
United States 269 282
January 286 293
However 300 307
Critics 386 393
North 413 418
Kissinger 434 443
Those 518 523
North 539 544
Le Duc Tho 653 663
Yasser Arafat 691 704
Shimon Peres 706 718
Yitzhak Rabin 724 737
Peace Prize 751 762
Israel 813 819
Palestine 824 833
Immediately 835 846
Norwegian Nobel Committee 894 919
Arafat 938 944
Additional 974 984
Arafat 1002 1008
Another 1054 1061
Peace Prize 1076 1087
Barack Obama 1108 1120
Nominations 1130 1141
Obama 1176 1181
President 1197 1206
United States 1214 1227
Obama 1292 1297
Past Peace Prize 1412 1428
Obama 1470 1475
Obama 1581 1586
Peace Prizes 1620 1632
Jimmy Carter 1637 1649
Al Gore 1654 1661


**Exercise 3.** First install spaCy: https://spacy.io/usage  using the link we provided, according to your operating system. Perform entity recognition using spaCy https://spacy.io/usage/linguistic-features#named-entities on the file example-nobel.txt . Compute the F1 score of the techniques from Exercise 1, 2 and of spaCy on the task of finding entities in the file example-nobel.txt. Note that for this you need to create the gold standard, i.e. the correct annotations. Do not take into account the entities of type DATE returned by spaCy.  

In [ ]:
# Solution exercise 3 (simple)

from pathlib import Path
import spacy


def f1(pred_set, gold_set):
	pred_set, gold_set = set(pred_set), set(gold_set)
	tp = len(pred_set & gold_set)
	fp = len(pred_set - gold_set)
	fn = len(gold_set - pred_set)
	p = tp / (tp + fp) if tp + fp else 0.0
	r = tp / (tp + fn) if tp + fn else 0.0
	f = 2 * p * r / (p + r) if p + r else 0.0
	return p, r, f, tp, fp, fn


text = Path("data/example-nobel.txt").read_text(encoding="utf-8")

# Gold standard (unique entity names in this short text)
GOLD = {
	"Henry Kissinger",
	"Le Duc Tho",
	"Norwegian Nobel Committee",
	"North Vietnam",
	"United States",
	"Yasser Arafat",
	"Shimon Peres",
	"Yitzhak Rabin",
	"Israel",
	"Palestine",
	"Barack Obama",
	"Jimmy Carter",
	"Al Gore",
}

# Exercise 1 predictions (unique strings)
P1 = {d["entity"] for d in found}

# Exercise 2 predictions (unique strings)
# mentions are tuples: (entity, start, end)
P2 = {m[0] for m in mentions}


nlp = spacy.load("en_core_web_sm")

doc = nlp(text)
P3 = {e.text for e in doc.ents if e.label_ != "DATE"}

for name, P in [("Trie", P1), ("Regex", P2), ("spaCy", P3)]:
	p, r, f, tp, fp, fn = f1(P, GOLD)
	print(f"{name:5s} | P={p:.3f} R={r:.3f} F1={f:.3f} (tp={tp}, fp={fp}, fn={fn})")

Trie  | P=1.000 R=0.615 F1=0.762 (tp=8, fp=0, fn=5)
Regex | P=0.406 R=1.000 F1=0.578 (tp=13, fp=19, fn=0)
spaCy | P=0.407 R=0.846 F1=0.550 (tp=11, fp=16, fn=2)


**Exercise 4.** Relations are not known. The ReVerb algorithm [Fader et al., 2011] has two steps: relation extraction and argument extraction.

A relation is of the form V|VP|VW*P where

- V=(aux verb|verb) particle? adverb?
- W=(noun|adj|adv|pron|det)
- P=(prep|particle)

In order to find the parts of speech needed for the algorithm, check https://spacy.io/usage/linguistic-features#pos-tagging . Note that spaCy allows you to add regular expressions easily, such that you can implement ReVerb: https://spacy.io/usage/rule-based-matching . Make sure to first split the text into sentences such that you can correctly identify the subject and object, by adapting the following code:

In [38]:
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("sentencizer")

doc = nlp("Paris is the capital of France. Paris is located in France.")
for sentence in doc.sents:
    print(sentence)
    print("Noun phrases:", [(chunk.text, chunk.start, chunk.end) for chunk in sentence.noun_chunks])

Paris is the capital of France.
Noun phrases: [('Paris', 0, 1), ('the capital', 2, 4), ('France', 5, 6)]
Paris is located in France.
Noun phrases: [('Paris', 7, 8), ('France', 11, 12)]


Here is an example of the first part of the regular expression you need to write to get the predicate:

In [44]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)
pattern1 = [{"POS": "AUX", "OP": "?"}, {"POS": "VERB"}, 
            {"POS": "PART", "OP":"?"}, {"POS": "ADV", "OP":"?"}]

matcher.add("V", [pattern1], greedy='LONGEST')
doc = nlp("Paris is the capital of France.")
for token in doc:
    print(token.text, token.pos_)
matches = matcher(doc)
for match_id, start, end in matches:
    string_id = nlp.vocab.strings[match_id]  # Get string representation
    span = doc[start:end]  # The matched span
    print(string_id, start, end, span.text)

Paris PROPN
is AUX
the DET
capital NOUN
of ADP
France PROPN
. PUNCT


Finish the code by adding the full regular expression from the class, V|VP|VW*P.
First try your code on a small paragraph, e.g. on the sentences above you get the triples (Paris, is the capital of, France) and (Paris, is located in , France). Then test your algorithm on a sample from Wikipedia, contained in the file *simplewiki.txt*.

Note that you can use part-of-speech tags to improve your NER algorithm that uses regular expression on capital letter (Exercise 2). Filter out words that are not nouns from your matches. 

In [ ]:
# Solution exercise 4

from pathlib import Path

import spacy
from spacy.matcher import Matcher


nlp = spacy.load("en_core_web_sm")
if "sentencizer" not in nlp.pipe_names:
	nlp.add_pipe("sentencizer", first=True)

matcher = Matcher(nlp.vocab)

V = [
	{"POS": "AUX", "OP": "?"},
	{"POS": "VERB"},
	{"POS": "PART", "OP": "?"},
	{"POS": "ADV", "OP": "?"},
]
W_star = [{"POS": {"IN": ["NOUN", "ADJ", "ADV", "PRON", "DET"]}, "OP": "*"}]
P = [{"POS": {"IN": ["ADP", "PART"]}}]

patterns = [
	V,  # V
	V + P,  # VP
	V + W_star + P,  # VW*P
]
matcher.add("PRED", patterns, greedy="LONGEST")


def extract_triples(text: str):
	doc = nlp(text)
	triples = []
	for sent in doc.sents:
		chunks = [c for c in doc.noun_chunks if c.start >= sent.start and c.end <= sent.end]
		if len(chunks) < 2:
			continue

		# take the longest predicate match fully inside the sentence
		pred = None
		for _, s, e in matcher(doc):
			if sent.start <= s and e <= sent.end:
				pred = doc[s:e].text
				break

		if pred:
			triples.append((chunks[0].text, pred, chunks[-1].text))
	return triples


print(extract_triples("Paris is the capital of France. Paris is located in France."))

wiki = Path("data/simplewiki.txt")
if wiki.exists():
	sample = "\n".join(wiki.read_text(encoding="utf-8").splitlines()[:40])
	print(extract_triples(sample)[:25])
else:
	print("data/simplewiki.txt not found")

[('Paris', 'is located in', 'France')]
[('The name', 'comes from', 'which'), ('This', 'growing plants in', 'spring'), ('April', 'begins on', 'leap years'), ('Poets', 'mean the end of', 'winter'), ('example', 'bring', 'May flowers'), ('It', 'has', '31 days'), ('This month', 'called', 'the old Roman calendar'), ('The Roman calendar', 'began in', 'Romulus'), ('It', 'were added to', 'King Numa Pompilius'), ('those two months', 'were moved from', '(Roman writers'), ('August', 'is named for', 'Augustus Caesar'), ('The month', 'wanted as many days as', "Julius Caesar's month"), ('An extra month', 'was moved from', 'August'), ('This', 'was undone', 'the leap year adjustment'), (' \nArt\tThe word', 'regarding an attraction to', 'the human senses'), ('art', 'is made', 'herself'), ('Some art', 'can put things in', 'things'), ('Many people', 'disagree on', 'art'), ('Many people', 'are driven to', 'their inner creativity'), ('Art', 'includes', 'theatre'), ('Some people', 'stimulating the human sens

**Exercise 5.** Filter the triples you extracted  such that you keep only the ones for which the subject is a named entity. 
You might have several mentions of the same entity, perform deduplication on these mentions (i.e. determine which mentions refer to the same object). For this, write a simple algorithm that given two entities in input, computes if they are the same using their names, and the label of the entities that is given when you run the named entity function of spaCy. If you find that mentions of entities with the same name that refer to different objects, modify the mention's name such that the mentions are distinct (eg: Europe_continent, Europe_person).  
Create a graph in NetworkX using the triples your obtain and find the important nodes in this graph, using different measures of importance that we have seen during the first course.

In [ ]:
# Solution exercice 5
